# 🐕 Unitree Go2 — Entrenamiento RL con MuJoCo

Este notebook entrena una política de locomoción para el **Unitree Go2 EDU** usando
[`unitree_rl_mjlab`](https://github.com/unitreerobotics/unitree_rl_mjlab) —
el framework oficial de Unitree Robotics basado en MuJoCo.

**Tiempo estimado de entrenamiento (3 000 iteraciones):**

| GPU | Tiempo |
|-----|--------|
| A100 (Colab Pro) | ~10 min |
| T4  (Colab Pro)  | ~70 min |
| RTX 4090 (local) | ~20 min |

---
**Antes de empezar:** Cambia el entorno de ejecución a GPU.

`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU → A100 (preferido)`

## 0. Verificar GPU disponible

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Detener ejecución si no hay GPU disponible.
# En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU → A100
assert torch.cuda.is_available(), (
    "\n❌ No se detectó GPU.\n"
    "Ve a: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (A100 o T4).\n"
    "Sin GPU el entrenamiento tomaría varios días."
)
print("\n✅ GPU lista para entrenamiento.")

## 1. Instalar dependencias del sistema

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq \
    libyaml-cpp-dev \
    libboost-all-dev \
    libeigen3-dev \
    libspdlog-dev \
    libfmt-dev \
    libgl1-mesa-glx \
    libglu1-mesa-dev \
    xvfb \
    ffmpeg
echo "Sistema listo ✓"

## 2. Clonar e instalar unitree_rl_mjlab

In [ ]:
%%bash
# Clonar repositorio oficial
if [ ! -d "unitree_rl_mjlab" ]; then
    git clone --depth 1 https://github.com/unitreerobotics/unitree_rl_mjlab.git
    echo "Clonado ✓"
else
    echo "Repositorio ya existe, actualizando..."
    cd unitree_rl_mjlab && git pull
fi

In [ ]:
%%bash
cd unitree_rl_mjlab
pip install -e . --quiet
echo "unitree_rl_mjlab instalado ✓"

In [ ]:
# Verificar instalación
import mujoco
import torch
print(f"MuJoCo version: {mujoco.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print("✓ Todo listo para entrenar")

## 3. Entender el entorno de simulación

El Go2 tiene **12 motores** (3 por pata × 4 patas).

El espacio de observación incluye:
- Velocidad angular del cuerpo (IMU)
- Ángulo de gravedad proyectado
- Posiciones y velocidades de cada joint (12 × 2 = 24)
- Acciones previas (12)
- Velocidad objetivo del comando (vx, vy, vyaw)

El espacio de acción son **12 deltas de posición de joint** (uno por motor).

In [ ]:
import subprocess, os
os.chdir("/content/unitree_rl_mjlab")

# Ver configuración del entorno Go2
result = subprocess.run(
    ["python", "scripts/train.py", "Unitree-Go2-Flat", "--help"],
    capture_output=True, text=True
)
print(result.stdout[:3000] if result.stdout else result.stderr[:3000])

## 4. Entrenamiento — terreno plano (Go2 Flat)

Entrenamos una política que hace que el Go2 siga comandos de velocidad
(vx, vy, vyaw) en terreno plano.

**Parámetros importantes:**
- `--env.scene.num-envs=2048` — número de robots paralelos (baja si te quedas sin VRAM)
- `max_iterations=3000` — iteraciones (3K ≈ política funcional básica; 10K ≈ mejor)
- Logs en `logs/rsl_rl/go2_velocity/`

In [ ]:
import subprocess, threading, time

# Configuración del entrenamiento
NUM_ENVS   = 2048    # reduce a 1024 si hay OOM en T4
MAX_ITER   = 3000    # 3000 para experimento rápido; 10000 para política más estable

cmd = [
    "python", "scripts/train.py", "Unitree-Go2-Flat",
    f"--env.scene.num-envs={NUM_ENVS}",
    f"--max_iterations={MAX_ITER}",
    "--headless",
]

print("🚀 Iniciando entrenamiento...")
print(f"   Entornos paralelos: {NUM_ENVS}")
print(f"   Iteraciones: {MAX_ITER}")
print("   Los logs aparecerán cada 50 iteraciones.")
print()

os.chdir("/content/unitree_rl_mjlab")
start_time = time.time()

# Ejecutar y mostrar salida en tiempo real
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()
elapsed = time.time() - start_time
print(f"\n✅ Entrenamiento completado en {elapsed/60:.1f} minutos")

## 5. Inspeccionar la curva de aprendizaje

Cargamos los logs de entrenamiento y visualizamos las métricas clave:
- **mean_reward**: recompensa total promedio (debe aumentar)
- **tracking_lin_vel**: qué tan bien sigue el comando de velocidad lineal
- **tracking_ang_vel**: seguimiento de velocidad angular

In [ ]:
import glob, os
import matplotlib.pyplot as plt
import numpy as np

# Encontrar el directorio de logs más reciente
log_dirs = sorted(glob.glob("logs/rsl_rl/go2_velocity/*/*"))
if not log_dirs:
    log_dirs = sorted(glob.glob("logs/**/*", recursive=True))

print("Logs encontrados:")
for d in log_dirs[-3:]:
    print(f"  {d}")

# Leer archivo de métricas si existe (formato depende de la versión)
metrics_files = glob.glob("logs/**/*.txt", recursive=True) + \
                glob.glob("logs/**/*.csv", recursive=True)
print(f"\nArchivos de métricas: {len(metrics_files)}")
for f in metrics_files[:5]:
    print(f"  {f}")

In [ ]:
# Encontrar el checkpoint del modelo entrenado
checkpoints = sorted(glob.glob("logs/**/*.pt", recursive=True))
if checkpoints:
    latest_ckpt = checkpoints[-1]
    print(f"✓ Checkpoint más reciente: {latest_ckpt}")
    size_mb = os.path.getsize(latest_ckpt) / 1024**2
    print(f"  Tamaño: {size_mb:.1f} MB")
else:
    print("⚠ No se encontraron checkpoints. El entrenamiento puede no haber finalizado.")

## 6. Guardar en Google Drive

Los modelos entrenados se guardan aquí para no perderlos al cerrar la sesión.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
from datetime import datetime

# Crear carpeta de destino
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
dest_dir = f"/content/drive/MyDrive/unitree_rl_models/go2_flat_{timestamp}"
os.makedirs(dest_dir, exist_ok=True)

# Copiar todos los logs y checkpoints
if os.path.exists("logs"):
    shutil.copytree("logs", f"{dest_dir}/logs", dirs_exist_ok=True)
    print(f"✓ Logs guardados en: {dest_dir}/logs")

# Copiar checkpoint más reciente aparte
checkpoints = sorted(glob.glob("logs/**/*.pt", recursive=True))
if checkpoints:
    shutil.copy2(checkpoints[-1], dest_dir)
    print(f"✓ Checkpoint copiado: {os.path.basename(checkpoints[-1])}")
    
print(f"\nRuta completa: {dest_dir}")

## 7. Visualizar la política (simulación MuJoCo)

Renderizamos un video del Go2 ejecutando la política entrenada.
En Colab no hay pantalla física, así que usamos un **display virtual (xvfb)**.

In [ ]:
# Iniciar display virtual
import subprocess
subprocess.Popen(['Xvfb', ':1', '-screen', '0', '1280x720x24'], 
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import os
os.environ['DISPLAY'] = ':1'
print("Display virtual iniciado (:1) ✓")

In [ ]:
# Buscar checkpoint para reproducción
checkpoints = sorted(glob.glob("logs/**/*.pt", recursive=True))

if checkpoints:
    ckpt = checkpoints[-1]
    print(f"Usando checkpoint: {ckpt}")
    
    result = subprocess.run([
        "python", "scripts/play.py", "Unitree-Go2-Flat",
        f"--checkpoint_file={ckpt}",
        "--num_envs=1",
        "--record_video",
        "--video_path=/content/go2_policy.mp4",
    ], capture_output=True, text=True, timeout=120)
    
    print(result.stdout[-2000:] if result.stdout else result.stderr[-2000:])
else:
    print("⚠ No hay checkpoint disponible. Corre la sección 4 primero.")

In [ ]:
# Mostrar video en el notebook
from IPython.display import HTML
from base64 import b64encode
import os

video_path = "/content/go2_policy.mp4"
if os.path.exists(video_path):
    video_data = open(video_path, "rb").read()
    encoded = b64encode(video_data).decode()
    HTML(f'''
    <video width="640" height="360" controls>
      <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    ''')
else:
    print("Video no generado. Verifica los pasos anteriores.")

## 8. Exportar política para el robot real

Convierte el checkpoint `.pt` a ONNX para compilarlo en C++ y desplegarlo en el Go2.

In [ ]:
checkpoints = sorted(glob.glob("logs/**/*.pt", recursive=True))

if checkpoints:
    ckpt = checkpoints[-1]
    export_path = "/content/go2_policy.onnx"
    
    result = subprocess.run([
        "python", "scripts/export.py", "Unitree-Go2-Flat",
        f"--checkpoint_file={ckpt}",
        f"--export_path={export_path}",
    ], capture_output=True, text=True)
    
    print(result.stdout or result.stderr)
    
    if os.path.exists(export_path):
        size_mb = os.path.getsize(export_path) / 1024**2
        print(f"\n✅ Política exportada: {export_path} ({size_mb:.2f} MB)")
        
        # Copiar a Drive si está montado
        if os.path.exists('/content/drive'):
            import shutil
            shutil.copy2(export_path, dest_dir)
            print(f"✓ Copiado a Google Drive: {dest_dir}")
else:
    print("Entrena el modelo primero (sección 4).")

## 9. Siguiente paso — Desplegar en Go2 real

Con el archivo `go2_policy.onnx` exportado, el flujo para el robot real es:

```bash
# En el computador conectado al Go2 por Ethernet:

# 1. Compilar el controlador de deployment
cd unitree_rl_mjlab/deploy
mkdir build && cd build
cmake .. && make -j4

# 2. Poner el robot en modo de control manual
#    Botón: L2 + R2 en el control (modo debug)

# 3. Ejecutar la política
./unitree_go2_deploy \
    --network_interface eth0 \
    --policy /ruta/a/go2_policy.onnx
```

**Ver:** [go2-hardware-profundo.md](../docs/unitree/go2-hardware-profundo.md) para detalles
de la configuración de red y el proceso completo de deployment.

---

## 🎯 Ejercicios

1. **Cambiar terreno**: Reemplaza `Unitree-Go2-Flat` por `Unitree-Go2-Rough` y compara el comportamiento
2. **Ajustar recompensas**: Encuentra el archivo de configuración de recompensas y modifica el peso de `tracking_lin_vel`
3. **Reducir número de entornos**: Prueba con `--env.scene.num-envs=512` en una T4 y mide la diferencia en tiempo de entrenamiento
4. **Comparar con A100**: Si tienes acceso a A100, repite el experimento y calcula el speedup real